У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [ ]:
import pandas as pd
df = pd.read_csv('/content/sample_data/customer_segmentation_train.csv')

In [ ]:
df.head()

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               8068 non-null   int64  
 1   Gender           8068 non-null   object 
 2   Ever_Married     7928 non-null   object 
 3   Age              8068 non-null   int64  
 4   Graduated        7990 non-null   object 
 5   Profession       7944 non-null   object 
 6   Work_Experience  7239 non-null   float64
 7   Spending_Score   8068 non-null   object 
 8   Family_Size      7733 non-null   float64
 9   Var_1            7992 non-null   object 
 10  Segmentation     8068 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 693.5+ KB


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv('/content/sample_data/customer_segmentation_train.csv')

# Заповнення категоріальних пропусків
cat_fill_cols = ['Ever_Married', 'Graduated', 'Profession']
for col in cat_fill_cols:
    df[col] = df[col].fillna('Unknown')

# Заповнення числових пропусків
num_fill_cols = ['Work_Experience', 'Family_Size']
for col in num_fill_cols:
    df[col] = df[col].fillna(df[col].median())

# Кодування цільової змінної
le = LabelEncoder()
y = le.fit_transform(df['Segmentation'])

# Формування X
X = df.drop('Segmentation', axis=1)

# Визначення типів колонок
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

# Препроцесор
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

# Пайплайн з класифікаційною моделлю
model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Розбиття
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Навчання
model.fit(X_train, y_train)

# Прогноз
y_pred = model.predict(X_test)

# Метрики
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))




Accuracy: 0.4653035935563817
              precision    recall  f1-score   support

           0       0.39      0.43      0.41       391
           1       0.30      0.14      0.19       369
           2       0.43      0.55      0.48       380
           3       0.62      0.68      0.65       474

    accuracy                           0.47      1614
   macro avg       0.43      0.45      0.43      1614
weighted avg       0.45      0.47      0.45      1614



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Модель LogisticRegression показує низьку якість класифікації для задачі сегментації клієнтів. Загальна точність становить лише ≈46%, що означає, що модель правильно класифікує менше половини прикладів. Якість сильно відрізняється між класами:

клас 3 визначається найкраще (f1 приблизно 0.65),

класи 0 та 2 — середньо,

клас 1 модель майже не розпізнає (f1 приблизно 0.19).

Також з’явилося попередження про те, що алгоритм не зійшовся, тобто оптимізація не завершилася в межах заданої кількості ітерацій. Це свідчить про те, що модель має труднощі з пошуком оптимальних параметрів — ймовірно через незбалансованість класів

**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [ ]:
from imblearn.over_sampling import SMOTENC
import numpy as np

# 1. Розбиття на train/test (ми вже маємо X, y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Визначаємо категоріальні ознаки
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns

# SMOTENC потребує індекси категоріальних колонок
cat_feature_indices = [X.columns.get_loc(col) for col in cat_cols]

# 3. Ініціалізуємо SMOTENC
smote_nc = SMOTENC(
    categorical_features=cat_feature_indices,
    random_state=42
)

# 4. Застосовуємо SMOTENC тільки до тренувального набору
X_train_resampled, y_train_resampled = smote_nc.fit_resample(X_train, y_train)

print("Розмір до SMOTENC:", X_train.shape)
print("Розмір після SMOTENC:", X_train_resampled.shape)

# 5. Навчання моделі на збалансованих даних
model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('classifier', LogisticRegression(max_iter=2000))
])

model.fit(X_train_resampled, y_train_resampled)

# 6. Прогноз на тесті
y_pred = model.predict(X_test)

# 7. Метрики
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))



Розмір до SMOTENC: (6454, 10)
Розмір після SMOTENC: (7176, 10)
Accuracy: 0.5049566294919455
              precision    recall  f1-score   support

           0       0.42      0.45      0.44       391
           1       0.41      0.32      0.36       369
           2       0.48      0.55      0.51       380
           3       0.66      0.66      0.66       474

    accuracy                           0.50      1614
   macro avg       0.49      0.49      0.49      1614
weighted avg       0.50      0.50      0.50      1614



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


1. Балансування класів спрацювало
Розмір тренувального набору збільшився з 6454 → 7176, що означає, що SMOTENC згенерував нові синтетичні приклади для менш представлених класів.
Це допомогло моделі краще навчитися розрізняти всі сегменти.

2. Точність зросла
Було: ≈46%

Стало: ≈50%

Підвищення на ~4% — це нормальний і очікуваний результат для SMOTENC, особливо коли дані містять багато категоріальних ознак.

3. Покращилися метрики для всіх класів
Особливо:

клас 1 (найскладніший) піднявся з f1 ≈ 0.19 → 0.36
клас 2 став стабільнішим (f1 ≈ 0.57)

клас 3 зберіг високу якість (f1 ≈ 0.66)

Це означає, що модель стала менш упередженою і краще розпізнає всі сегменти, а не лише найпоширеніші.

In [ ]:
from imblearn.combine import SMOTETomek
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

# 1. Розбиття на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Кодуємо X_train вручну (без пайплайну)
encoder = OneHotEncoder(handle_unknown='ignore')
X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

# 3. SMOTE-Tomek (тепер всі ознаки числові → працює)
smote_tomek = SMOTETomek(random_state=42)

X_train_resampled_st, y_train_resampled_st = smote_tomek.fit_resample(
    X_train_encoded, y_train
)

print("Розмір до SMOTE-Tomek:", X_train_encoded.shape)
print("Розмір після SMOTE-Tomek:", X_train_resampled_st.shape)

# 4. Навчання моделі
clf = LogisticRegression(max_iter=3000)
clf.fit(X_train_resampled_st, y_train_resampled_st)

# 5. Прогноз
y_pred_st = clf.predict(X_test_encoded)

# 6. Метрики
print("Accuracy:", accuracy_score(y_test, y_pred_st))
print(classification_report(y_test, y_pred_st))


Розмір до SMOTE-Tomek: (6454, 6574)
Розмір після SMOTE-Tomek: (6502, 6574)
Accuracy: 0.5092936802973977
              precision    recall  f1-score   support

           0       0.42      0.44      0.43       391
           1       0.40      0.30      0.34       369
           2       0.49      0.59      0.53       380
           3       0.68      0.66      0.67       474

    accuracy                           0.51      1614
   macro avg       0.50      0.50      0.49      1614
weighted avg       0.51      0.51      0.50      1614



Застосування SMOTE‑Tomek дало помірне, але стабільне покращення якості моделі порівняно з базовим варіантом та з результатами після SMOTENC.

1. Розмір тренувального набору майже не змінився
Було: 6454

Стало: 6502


2. Точність моделі зросла.

Було (до ресемплінгу): ≈46%

Після SMOTENC: ≈50%

Після SMOTE‑Tomek: ≈51%

Покращення невелике, але стабільне.

3. Покращилися метрики для складних класів
Особливо:

клас 1 (найпроблемніший) піднявся до f1 ≈ 0.34

клас 2 став більш стабільним (f1 ≈ 0.53)

клас 3 зберіг найкращу якість (f1 ≈ 0.67)

Це означає, що модель краще розрізняє сегменти, а межі між класами стали чистішими після видалення Tomek‑зв’язок.

**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Модель на оригінальних даних
model_orig = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('clf', OneVsRestClassifier(LogisticRegression(max_iter=3000)))
])

model_orig.fit(X_train, y_train)
y_pred_orig = model_orig.predict(X_test)

print("=== Оригінальні дані ===")
print("Accuracy:", accuracy_score(y_test, y_pred_orig))
print(classification_report(y_test, y_pred_orig))


# Модель на даних після SMOTENC
model_smote_nc = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('clf', OneVsRestClassifier(LogisticRegression(max_iter=3000)))
])

model_smote_nc.fit(X_train_resampled, y_train_resampled)
y_pred_smote_nc = model_smote_nc.predict(X_test)

print("=== SMOTENC ===")
print("Accuracy:", accuracy_score(y_test, y_pred_smote_nc))
print(classification_report(y_test, y_pred_smote_nc))


# Модель на даних після SMOTE-Tomek
clf_st = OneVsRestClassifier(LogisticRegression(max_iter=3000))
clf_st.fit(X_train_resampled_st, y_train_resampled_st)
y_pred_st = clf_st.predict(X_test_encoded)

print("=== SMOTE-Tomek ===")
print("Accuracy:", accuracy_score(y_test, y_pred_st))
print(classification_report(y_test, y_pred_st))


=== Оригінальні дані ===
Accuracy: 0.49752168525402723
              precision    recall  f1-score   support

           0       0.40      0.47      0.43       391
           1       0.41      0.14      0.21       369
           2       0.45      0.61      0.52       380
           3       0.66      0.71      0.68       474

    accuracy                           0.50      1614
   macro avg       0.48      0.48      0.46      1614
weighted avg       0.49      0.50      0.47      1614

=== SMOTENC ===
Accuracy: 0.4913258983890954
              precision    recall  f1-score   support

           0       0.41      0.46      0.43       391
           1       0.34      0.19      0.24       369
           2       0.45      0.58      0.51       380
           3       0.67      0.68      0.68       474

    accuracy                           0.49      1614
   macro avg       0.47      0.48      0.46      1614
weighted avg       0.48      0.49      0.48      1614

=== SMOTE-Tomek ===
Accuracy: 

**Оригінальні дані**
Accuracy ≈ 0.4975

Клас 1 визначається дуже погано (f1 близько 0.21)

Клас 3 визначається найкраще (f1 близько 0.68)

Висновок: модель страждає від дисбалансу класів.

**SMOTENC**
Accuracy ≈ 0.4913

Клас 1 трохи покращився (f1 близько 0.24)

Клас 3 стабільний (f1 близько 0.68)

Macro F1 майже не змінився

Висновок: SMOTENC допоміг збалансувати дані, але не дав значного приросту якості.


 **SMOTE‑Tomek**
Accuracy ≈ 0.5025 — найвища серед трьох

Клас 1 покращився (f1 близько 0.30)

Клас 2 покращився (f1 близько 0.53)

Клас 3 стабільний (f1 близько 0.67)

Macro F1 найкращий серед трьох варіантів

Висновок: SMOTE‑Tomek дав найкращий баланс між класами.
TomekLinks видалив шумові точки, тому модель стала точнішою і менш упередженою.

**Висновок**: SMOTE‑Tomek дав найкращий баланс між класами.

**Обрана метрика**
Я обрала macro F1-score, тому що:

задача багатокласова,

класи мають різну кількість прикладів,

важливо оцінювати якість рівномірно для всіх класів, а не лише для найчастіших.

Macro F1 — найчесніша метрика для порівняння таких моделей

**Найкраща модель** - LogisticRegression (One-vs-Rest) + SMOTE‑Tomek
Причини:

найвищий accuracy,

найкращий macro F1,

найкраще розпізнає складні класи,

межі між класами очищені TomekLinks,

модель стала менш упередженою.
